In [1]:
!pip install "numpy<2" -qqq
!pip install surprise -qqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 24.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
p

In [ ]:
import pandas as pd;import polars as pl; import numpy as np; from surprise import Dataset, Reader, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from surprise import KNNBasic, KNNWithMeans, KNNWithZScore, KNNBaseline
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import gdown, zipfile
url = 'https://drive.google.com/file/d/1HbSTBkFeczfTSkzm3KCIwzy6fOouTg_i/view?usp=sharing'
path = gdown.download(url, quiet=False, fuzzy=True)
with zipfile.ZipFile(path, 'r') as f: f.extractall('data')

#1: Подготовка данных

In [ ]:
r_l = pl.scan_csv('/content/data/ml-latest-small/ratings.csv').select(['userId', 'movieId', 'rating'])
m_l = pl.scan_csv('/content/data/ml-latest-small/movies.csv')
df = r_l.collect(); mov = m_l.collect()
r = Reader(rating_scale=(1, 5)); ds = Dataset.load_from_df(df.to_pandas(), r)
tr_set, tst_set = train_test_split(ds, test_size=0.2)

#2: Коллаборативная фильтрация (KNN)

In [ ]:
knn_variants = {
    'KNNBasic': KNNBasic(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNWithMeans': KNNWithMeans(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNWithZScore': KNNWithZScore(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNBaseline': KNNBaseline(sim_options={'name': 'cosine', 'user_based': True})
}

knn_metrics = []

for name, algo in knn_variants.items():
    algo.fit(tr_set)
    predictions = algo.test(tst_set)

    y_t_knn = [p.r_ui for p in predictions]
    y_p_knn = [p.est for p in predictions]

    metrics = [
        r2_score(y_t_knn, y_p_knn),
        np.sqrt(mean_squared_error(y_t_knn, y_p_knn)),
        mean_absolute_error(y_t_knn, y_p_knn),
        mean_squared_error(y_t_knn, y_p_knn)
    ]
    knn_metrics.append(metrics)

res_knn = pd.DataFrame(knn_metrics, columns=['R2', 'RMSE', 'MAE', 'MSE'], index=knn_variants.keys())
display(res_knn.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
                     .background_gradient(cmap='RdYlGn', subset=['R2']))

best_knn_name = res_knn['R2'].idxmax()
print(f"Лучшая модель для гибрида: {best_knn_name}")

best_algo = knn_variants[best_knn_name]
best_pred = best_algo.test(tst_set)
y_p = [p.est for p in best_pred]
met_cf = knn_metrics[list(knn_variants.keys()).index(best_knn_name)]

y_t = [p[2] for p in tst_set] # p[2] — это оценка (rating) в кортеже (uid, iid, rating)



Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.


,R2,RMSE,MAE,MSE
KNNBasic,0.138023,0.969451,0.746683,0.939834
KNNWithMeans,0.255758,0.900813,0.689559,0.811465
KNNWithZScore,0.262255,0.896873,0.682274,0.804381
KNNBaseline,0.294693,0.876934,0.671601,0.769013


Лучшая модель для гибрида: KNNBaseline


#3: Контентная фильтрация (жанры)

In [ ]:
#3. Реализация контентной фильтрации
g_e = mov.with_columns(pl.col("genres").str.split("|")).explode("genres") \
         .pivot(index="movieId", on="genres", values="genres", aggregate_function="len").fill_null(0)
#Приводим к Int8 только столбцы жанров (кроме movieId)
cols = [c for c in g_e.columns if c != 'movieId']
g_e = g_e.with_columns([pl.col(c).cast(pl.Int8) for c in cols])

# 1. Считаем среднее по каждому пользователю
user_means = df.group_by("userId").agg(pl.col("rating").mean().alias("avg_rating"))
df_norm = df.join(user_means, on="userId").with_columns(
    (pl.col("rating") - pl.col("avg_rating")).alias("norm_rating")
)

# 2. Создаем профили пользователей на основе нормализованных оценок
u_p = df_norm.join(g_e, on="movieId").group_by("userId").agg(
    # Взвешиваем жанры фильма на отклонение оценки от среднего
    [(pl.col(c) * pl.col("norm_rating")).sum() / (pl.col("norm_rating").std() + 1e-9)
     for c in cols]
)

u_d = dict(zip(u_p["userId"].to_list(), u_p.drop("userId").to_numpy()))
m_d = dict(zip(g_e["movieId"].to_list(), g_e.drop("movieId").to_numpy()))

# 3. Предикт (теперь результат будет колебаться вокруг 0)
def cb_p(u, m):
    try:
        # Считаем схожесть векторов
        u_vec = u_d[u]
        m_vec = m_d[m]
        # Скалярное произведение нормализованных векторов
        score = np.dot(u_vec, m_vec) / (np.linalg.norm(u_vec) * np.linalg.norm(m_vec) + 1e-9)
        # Возвращаем к шкале 1-5, прибавляя среднее (например, 3.5)
        return 3.5 + score * 1.5
    except:
        return 3.5


pred_cb = [cb_p(p[0], p[1]) for p in tst_set];
y_p_cb = np.clip(pred_cb, 1, 5)
met_cb = [r2_score(y_t, y_p_cb), np.sqrt(mean_squared_error(y_t, y_p_cb)), mean_absolute_error(y_t, y_p_cb), mean_squared_error(y_t, y_p_cb)]
display(met_cb)

[0.023395784176757473,
 1.0318986407212298,
 0.8147844001241185,
 1.0648148047223216]

#4: Гибридизация и оценка

In [ ]:
# Теперь запускаем перебор alpha (ваш код ниже)
y_p_knn_arr = np.array(y_p)
y_p_cb_arr = np.array(y_p_cb)
y_t_arr = np.array(y_t)

alphas = np.linspace(0, 1, 21)
grid_data = []

for a in alphas:
    y_p_h_tmp = a * np.array(y_p) + (1 - a) * y_p_cb
    y_p_h_tmp = np.clip(y_p_h_tmp, 1, 5)
    grid_data.append([
        a,
        r2_score(y_t, y_p_h_tmp),
        np.sqrt(mean_squared_error(y_t, y_p_h_tmp)),
        mean_absolute_error(y_t, y_p_h_tmp),
        mean_squared_error(y_t, y_p_h_tmp)
    ])

# Вывод таблицы перебора для анализа
df_grid = pd.DataFrame(grid_data, columns=['alpha', 'R2', 'RMSE', 'MAE', 'MSE'])
display(df_grid.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
               .background_gradient(cmap='RdYlGn', subset=['R2'])
               .format({'alpha': '{:.2f}'}))

# 2. Фиксация лучшего 'a' на основе максимального R2
best_idx = df_grid['R2'].idxmax()
a = df_grid.loc[best_idx, 'alpha']

# 3. Финальный расчет с зафиксированным 'a'
y_p_h = a * np.array(y_p) + (1 - a) * y_p_cb
y_p_h = np.clip(y_p_h, 1, 5)

met_h = [
    r2_score(y_t, y_p_h),
    np.sqrt(mean_squared_error(y_t, y_p_h)),
    mean_absolute_error(y_t, y_p_h),
    mean_squared_error(y_t, y_p_h)
]

# 4. Итоговое сравнение моделей
res = pd.DataFrame([met_cf, met_cb, met_h],
                   columns=['R2', 'RMSE', 'MAE', 'MSE'],
                   index=['KNN', 'Content', 'Hybrid'])

print(f"\nЛучшее значение alpha зафиксировано: {a}")
display(res.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
                 .background_gradient(cmap='RdYlGn', subset=['R2']))


,alpha,R2,RMSE,MAE,MSE
0,0.00,0.023396,1.031899,0.814784,1.064815
1,0.05,0.064011,1.010213,0.796615,1.020531
2,0.10,0.101779,0.989622,0.779298,0.979352
3,0.15,0.136699,0.970195,0.762818,0.941278
4,0.20,0.168772,0.952002,0.747310,0.906308
5,0.25,0.197997,0.935116,0.732853,0.874443
6,0.30,0.224375,0.919610,0.719523,0.845682
7,0.35,0.247906,0.905553,0.707245,0.820026
8,0.40,0.268589,0.893014,0.696068,0.797474
9,0.45,0.286425,0.882059,0.686025,0.778027



Лучшее значение alpha зафиксировано: 0.75


,R2,RMSE,MAE,MSE
KNN,0.294693,0.876934,0.671601,0.769013
Content,0.023396,1.031899,0.814784,1.064815
Hybrid,0.333646,0.852374,0.655907,0.726542


#ИНДЗ

# Описание датасета: Goodreads Interactions Dataset

Данный датасет представляет собой репрезентативную выборку данных с платформы **Goodreads**. Он идеально подходит для построения рекомендательных систем с использованием как явной (explicit), так и неявной (implicit) обратной связи.

## 1. Основные характеристики
*   **Источник:** [Kaggle / arashnic](https://kaggle.com)
*   **Тип данных:** Табличные (CSV)
*   **Сфера:** Книжные рекомендации (Book Recommendations)
*   **Размер:** Содержит миллионы взаимодействий пользователей с книгами.

## 2. Структура файлов
Датасет обычно состоит из нескольких ключевых файлов (в зависимости от версии архива):

### Файл `interactions.csv` (или аналогичные по жанрам)
Это основной файл с транзакциями. Содержит следующие колонки:
*   `user_id`: Уникальный идентификатор пользователя (хэшированный).
*   `book_id`: Уникальный идентификатор книги.
*   `is_read`: Бинарный флаг (1 — прочитано, 0 — нет).
*   `rating`: **Явная обратная связь.** Оценка от 0 до 5.
    *   *Примечание:* 0 обычно означает, что пользователь добавил книгу, но не выставил оценку.
*   `is_reviewed`: Флаг наличия текстового отзыва.

### Файл `metadata.csv` (или `books.json/csv`)
Содержит информацию о самих книгах:
*   `book_id`: ID для связи с таблицей взаимодействий.
*   `title`: Название книги.
*   `authors`: Список авторов.
*   `average_rating`: Средний рейтинг на платформе.
*   `description`: Краткое описание (аннотация).
*   `genre`: Жанровая принадлежность.



In [20]:
import os
from google.colab import userdata

# Безопасное получение данных без проблем с кодировкой в коде
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!kaggle datasets download -d arashnic/goodreads-interactions-dataset --unzip

Dataset URL: https://www.kaggle.com/datasets/arashnic/goodreads-interactions-dataset
License(s): other
100% 184M/184M [00:01<00:00, 111MB/s]



In [3]:
import pandas as pd
import glob

# Поиск всех csv файлов и вывод их колонок
for file in glob.glob("*.csv"):
    df_head = pd.read_csv(file, nrows=0) # Читаем только заголовки
    print(f"Файл: {file}")
    print(f"Колонки: {df_head.columns.tolist()}\n")


Файл: articles.csv
Колонки: ['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']

Файл: books_metadata_small.csv
Колонки: ['book_id', 'average_rating', 'is_ebook', 'num_pages', 'publication_year', 'ratings_count', 'language_code']

Файл: interactions_small.csv
Колонки: ['user_id', 'book_id', 'is_read', 'rating']

Файл: customers.csv
Колонки: ['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']

Файл: transactions_train.csv
Колонки: ['t_dat', 'customer_id', 'article_id', 'price', 'sale

In [5]:
import os; import pandas as pd; import polars as pl; import numpy as np; from surprise import Dataset, Reader, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from surprise import KNNBasic, KNNWithZScore, KNNBaseline; from sklearn.metrics.pairwise import cosine_similarity
from google.colab import userdata

c_f = [f for f in os.listdir() if f.endswith('.csv')]
if not c_f: raise FileNotFoundError("CSV")

r_f = next((f for f in c_f if 'interact' in f.lower() or 'rating' in f.lower()), c_f[0])
m_f = next((f for f in c_f if 'book' in f.lower() and f != r_f), c_f[1])
print(f"Используемые файлы данных -> Рейтинги: {r_f} | Метаданные: {m_f}")

r_l = pl.scan_csv(r_f); m_l = pl.scan_csv(m_f)
n_r = r_l.collect_schema().names()
u_c = next(c for c in n_r if 'user' in c.lower())
i_c = next(c for c in n_r if 'book' in c.lower() or 'item' in c.lower())
r_c = next(c for c in n_r if 'rating' in c.lower())

df = r_l.filter(pl.col(r_c) > 0).select([u_c, i_c, r_c]).head(10000).collect()
r = Reader(rating_scale=(1, 5)); ds = Dataset.load_from_df(df.to_pandas(), r)
tr_set, tst_set = train_test_split(ds, test_size=0.2)

k_v = {'KNN Basic (Базовый)': KNNBasic(sim_options={'name': 'cosine', 'user_based': True}), 'KNN With Means (Со средними)': KNNWithMeans(sim_options={'name': 'cosine', 'user_based': True}), 'KNN With Z-Score (С нормализацией)': KNNWithZScore(sim_options={'name': 'cosine', 'user_based': True}), 'KNN Baseline (С оптимизацией смещений)': KNNBaseline(sim_options={'name': 'cosine', 'user_based': True})}
k_m = []

print("Этап 1: Обучение моделей коллаборативной фильтрации (KNN)...")
for n, a in k_v.items():
    print(f"Выполняется обучение алгоритма: {n}")
    a.fit(tr_set); pr = a.test(tst_set)
    y_t_k = [p.r_ui for p in pr]; y_p_k = [p.est for p in pr]
    k_m.append([r2_score(y_t_k, y_p_k), np.sqrt(mean_squared_error(y_t_k, y_p_k)), mean_absolute_error(y_t_k, y_p_k), mean_squared_error(y_t_k, y_p_k)])

r_k = pd.DataFrame(k_m, columns=['R-Squared', 'Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)'], index=k_v.keys())
display(r_k.style.background_gradient(cmap='RdYlGn_r', subset=['Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)']).background_gradient(cmap='RdYlGn', subset=['R-Squared']))

b_k_n = r_k['R-Squared'].idxmax(); print(f"Выбранная оптимальная модель коллаборативной фильтрации: {b_k_n}")
b_a = k_v[b_k_n]; b_pr = b_a.test(tst_set)
y_p = [p.est for p in b_pr]; m_cf = k_m[list(k_v.keys()).index(b_k_n)]
y_t = [p[2] for p in tst_set]

print("Этап 2: Извлечение признаков и обучение контентной фильтрации (Content-Based)...")
mov = m_l.head(10000).collect()
n_m = mov.columns
i_c_m = next(c for c in n_m if 'book' in c.lower() or 'item' in c.lower())
g_c = next((c for c in n_m if any(x in c.lower() for x in ['language', 'ebook', 'publication'])), None)
if not g_c: raise ValueError("Не найден целевой столбец для извлечения признаков")

g_e = mov.select([i_c_m, g_c]).with_columns(pl.col(g_c).cast(pl.Utf8).fill_null("UNK").str.split(",")).explode(g_c).pivot(index=i_c_m, on=g_c, values=g_c, aggregate_function="len").fill_null(0)
cols = [c for c in g_e.columns if c != i_c_m]
g_e = g_e.with_columns([pl.col(c).cast(pl.Int8) for c in cols])

u_m = df.group_by(u_c).agg(pl.col(r_c).mean().alias("a_r"))
df_n = df.join(u_m, on=u_c).with_columns((pl.col(r_c) - pl.col("a_r")).alias("n_r"))
u_p = df_n.join(g_e, left_on=i_c, right_on=i_c_m).group_by(u_c).agg([(pl.col(c) * pl.col("n_r")).sum() / (pl.col("n_r").std().fill_null(1.0) + 1e-9) for c in cols]).fill_null(0.0).fill_nan(0.0)

u_d = dict(zip(u_p[u_c].to_list(), u_p.drop(u_c).to_numpy()))
m_d = dict(zip(g_e[i_c_m].to_list(), g_e.drop(i_c_m).to_numpy()))

def c_p(u, m):
    try:
        u_v = u_d[u]; m_v = m_d[m]
        s = np.dot(u_v, m_v) / (np.linalg.norm(u_v) * np.linalg.norm(m_v) + 1e-9)
        return 3.5 + s * 1.5
    except:
        return 3.5

p_cb = [c_p(p[0], p[1]) for p in tst_set]; y_p_c = np.clip(p_cb, 1, 5)
m_cb = [r2_score(y_t, y_p_c), np.sqrt(mean_squared_error(y_t, y_p_c)), mean_absolute_error(y_t, y_p_c), mean_squared_error(y_t, y_p_c)]

print("Этап 3: Подбор оптимального коэффициента альфа для гибридизации...")
a_s = np.linspace(0, 1, 21); g_d = []
for a in a_s:
    y_p_h_t = a * np.array(y_p) + (1 - a) * y_p_c
    y_p_h_t = np.clip(y_p_h_t, 1, 5)
    g_d.append([a, r2_score(y_t, y_p_h_t), np.sqrt(mean_squared_error(y_t, y_p_h_t)), mean_absolute_error(y_t, y_p_h_t), mean_squared_error(y_t, y_p_h_t)])

d_g = pd.DataFrame(g_d, columns=['Альфа Коэффициент', 'R-Squared', 'Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)'])
display(d_g.style.background_gradient(cmap='RdYlGn_r', subset=['Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)']).background_gradient(cmap='RdYlGn', subset=['R-Squared']).format({'Альфа Коэффициент': '{:.2f}'}))

b_i = d_g['R-Squared'].idxmax(); a_v = d_g.loc[b_i, 'Альфа Коэффициент']
y_p_h = a_v * np.array(y_p) + (1 - a_v) * y_p_c; y_p_h = np.clip(y_p_h, 1, 5)

m_h = [r2_score(y_t, y_p_h), np.sqrt(mean_squared_error(y_t, y_p_h)), mean_absolute_error(y_t, y_p_h), mean_squared_error(y_t, y_p_h)]
res = pd.DataFrame([m_cf, m_cb, m_h], columns=['R-Squared', 'Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)'], index=['Коллаборативная модель (KNN)', 'Контентная модель (Content-Based)', 'Гибридная модель (Hybrid)'])
print(f"Этап 4: Итоговое сравнение. Зафиксирован лучший коэффициент Альфа: {a_v}")
display(res.style.background_gradient(cmap='RdYlGn_r', subset=['Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)']).background_gradient(cmap='RdYlGn', subset=['R-Squared']))

Используемые файлы данных -> Рейтинги: interactions_small.csv | Метаданные: books_metadata_small.csv
Этап 1: Обучение моделей коллаборативной фильтрации (KNN)...
Выполняется обучение алгоритма: KNN Basic (Базовый)
Computing the cosine similarity matrix...
Done computing similarity matrix.
Выполняется обучение алгоритма: KNN With Means (Со средними)
Computing the cosine similarity matrix...
Done computing similarity matrix.
Выполняется обучение алгоритма: KNN With Z-Score (С нормализацией)
Computing the cosine similarity matrix...
Done computing similarity matrix.
Выполняется обучение алгоритма: KNN Baseline (С оптимизацией смещений)
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.


,R-Squared,Root Mean Squared Error (RMSE),Mean Absolute Error (MAE),Mean Squared Error (MSE)
KNN Basic (Базовый),-0.101784,1.019605,0.800669,1.039595
KNN With Means (Со средними),-0.026018,0.983924,0.745566,0.968106
KNN With Z-Score (С нормализацией),-0.030744,0.986187,0.745588,0.972564
KNN Baseline (С оптимизацией смещений),0.053383,0.945085,0.749536,0.893186


Выбранная оптимальная модель коллаборативной фильтрации: KNN Baseline (С оптимизацией смещений)
Этап 2: Извлечение признаков и обучение контентной фильтрации (Content-Based)...
Этап 3: Подбор оптимального коэффициента альфа для гибридизации...


,Альфа Коэффициент,R-Squared,Root Mean Squared Error (RMSE),Mean Absolute Error (MAE),Mean Squared Error (MSE)
0,0.00,-0.591898,1.225579,1.024629,1.502045
1,0.05,-0.513441,1.194996,1.003513,1.428016
2,0.10,-0.439846,1.165579,0.982398,1.358575
3,0.15,-0.371114,1.137419,0.961282,1.293723
4,0.20,-0.307244,1.110611,0.940167,1.233458
5,0.25,-0.248236,1.085256,0.919051,1.177781
6,0.30,-0.194091,1.061458,0.897935,1.126692
7,0.35,-0.144809,1.039323,0.877036,1.080191
8,0.40,-0.100389,1.018960,0.857101,1.038278
9,0.45,-0.060831,1.000477,0.838158,1.000954


Этап 4: Итоговое сравнение. Зафиксирован лучший коэффициент Альфа: 0.8500000000000001


,R-Squared,Root Mean Squared Error (RMSE),Mean Absolute Error (MAE),Mean Squared Error (MSE)
Коллаборативная модель (KNN),0.053383,0.945085,0.749536,0.893186
Контентная модель (Content-Based),-0.591898,1.225579,1.024629,1.502045
Гибридная модель (Hybrid),0.080583,0.931408,0.753281,0.867522


### С нормализацией оценок

In [6]:
import os; import pandas as pd; import polars as pl; import numpy as np; from surprise import Dataset, Reader, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from surprise import KNNBasic, KNNWithZScore, KNNBaseline; from sklearn.metrics.pairwise import cosine_similarity
from google.colab import userdata


c_f = [f for f in os.listdir() if f.endswith('.csv')]
if not c_f: raise FileNotFoundError("CSV")

r_f = next((f for f in c_f if 'interact' in f.lower() or 'rating' in f.lower()), c_f[0])
m_f = next((f for f in c_f if 'book' in f.lower() and f != r_f), c_f[1])
print(f"Используемые файлы данных -> Рейтинги: {r_f} | Метаданные: {m_f}")

r_l = pl.scan_csv(r_f); m_l = pl.scan_csv(m_f)
n_r = r_l.collect_schema().names()
u_c = next(c for c in n_r if 'user' in c.lower())
i_c = next(c for c in n_r if 'book' in c.lower() or 'item' in c.lower())
r_c = next(c for c in n_r if 'rating' in c.lower())

print("Этап 0: Выполнение базовой предобработки (фильтрация пустот и выбор столбцов)...")
df = r_l.filter(pl.col(r_c) > 0).select([u_c, i_c, r_c]).head(10000).collect()
r = Reader(rating_scale=(1, 5)); ds = Dataset.load_from_df(df.to_pandas(), r)
tr_set, tst_set = train_test_split(ds, test_size=0.2)

k_v = {'KNN Basic (Базовый)': KNNBasic(sim_options={'name': 'cosine', 'user_based': True}), 'KNN With Means (Со средними)': KNNWithMeans(sim_options={'name': 'cosine', 'user_based': True}), 'KNN With Z-Score (С нормализацией)': KNNWithZScore(sim_options={'name': 'cosine', 'user_based': True}), 'KNN Baseline (С оптимизацией смещений)': KNNBaseline(sim_options={'name': 'cosine', 'user_based': True})}
k_m = []

print("Этап 1: Обучение моделей коллаборативной фильтрации (KNN)...")
for n, a in k_v.items():
    print(f"Выполняется обучение алгоритма: {n}")
    a.fit(tr_set); pr = a.test(tst_set)
    y_t_k = [p.r_ui for p in pr]; y_p_k = [p.est for p in pr]
    k_m.append([r2_score(y_t_k, y_p_k), np.sqrt(mean_squared_error(y_t_k, y_p_k)), mean_absolute_error(y_t_k, y_p_k), mean_squared_error(y_t_k, y_p_k)])

r_k = pd.DataFrame(k_m, columns=['R-Squared', 'Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)'], index=k_v.keys())
display(r_k.style.background_gradient(cmap='RdYlGn_r', subset=['Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)']).background_gradient(cmap='RdYlGn', subset=['R-Squared']))

b_k_n = r_k['R-Squared'].idxmax(); print(f"Выбранная оптимальная модель коллаборативной фильтрации: {b_k_n}")
b_a = k_v[b_k_n]; b_pr = b_a.test(tst_set)
y_p = [p.est for p in b_pr]; m_cf = k_m[list(k_v.keys()).index(b_k_n)]
y_t = [p[2] for p in tst_set]

print("Этап 2: Извлечение признаков и применение центрирования с Z-нормализацией...")
mov = m_l.head(10000).collect()
n_m = mov.columns
i_c_m = next(c for c in n_m if 'book' in c.lower() or 'item' in c.lower())
g_c = next((c for c in n_m if any(x in c.lower() for x in ['language', 'ebook', 'publication'])), None)
if not g_c: raise ValueError("Не найден целевой столбец для извлечения признаков")

g_e = mov.select([i_c_m, g_c]).with_columns(pl.col(g_c).cast(pl.Utf8).fill_null("UNK").str.split(",")).explode(g_c).pivot(index=i_c_m, on=g_c, values=g_c, aggregate_function="len").fill_null(0)
cols = [c for c in g_e.columns if c != i_c_m]
g_e = g_e.with_columns([pl.col(c).cast(pl.Int8) for c in cols])

print("Этап 2.1: Математическое вычисление средних значений и Z-оценок рейтингов...")
u_m = df.group_by(u_c).agg(pl.col(r_c).mean().alias("a_r"))
df_n = df.join(u_m, on=u_c).with_columns((pl.col(r_c) - pl.col("a_r")).alias("n_r"))
u_p = df_n.join(g_e, left_on=i_c, right_on=i_c_m).group_by(u_c).agg([(pl.col(c) * pl.col("n_r")).sum() / (pl.col("n_r").std().fill_null(1.0) + 1e-9) for c in cols]).fill_null(0.0).fill_nan(0.0)

u_d = dict(zip(u_p[u_c].to_list(), u_p.drop(u_c).to_numpy()))
m_d = dict(zip(g_e[i_c_m].to_list(), g_e.drop(i_c_m).to_numpy()))

def c_p(u, m):
    try:
        u_v = u_d[u]; m_v = m_d[m]
        s = np.dot(u_v, m_v) / (np.linalg.norm(u_v) * np.linalg.norm(m_v) + 1e-9)
        return 3.5 + s * 1.5
    except:
        return 3.5

p_cb = [c_p(p[0], p[1]) for p in tst_set]; y_p_c = np.clip(p_cb, 1, 5)
m_cb = [r2_score(y_t, y_p_c), np.sqrt(mean_squared_error(y_t, y_p_c)), mean_absolute_error(y_t, y_p_c), mean_squared_error(y_t, y_p_c)]

print("Этап 3: Выполнение гибридизации и поиск оптимального весового коэффициента Альфа...")
a_s = np.linspace(0, 1, 21); g_d = []
for a in a_s:
    y_p_h_t = a * np.array(y_p) + (1 - a) * y_p_c
    y_p_h_t = np.clip(y_p_h_t, 1, 5)
    g_d.append([a, r2_score(y_t, y_p_h_t), np.sqrt(mean_squared_error(y_t, y_p_h_t)), mean_absolute_error(y_t, y_p_h_t), mean_squared_error(y_t, y_p_h_t)])

d_g = pd.DataFrame(g_d, columns=['Альфа Коэффициент', 'R-Squared', 'Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)'])
display(d_g.style.background_gradient(cmap='RdYlGn_r', subset=['Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)']).background_gradient(cmap='RdYlGn', subset=['R-Squared']).format({'Альфа Коэффициент': '{:.2f}'}))

b_i = d_g['R-Squared'].idxmax(); a_v = d_g.loc[b_i, 'Альфа Коэффициент']
y_p_h = a_v * np.array(y_p) + (1 - a_v) * y_p_c; y_p_h = np.clip(y_p_h, 1, 5)

m_h = [r2_score(y_t, y_p_h), np.sqrt(mean_squared_error(y_t, y_p_h)), mean_absolute_error(y_t, y_p_h), mean_squared_error(y_t, y_p_h)]
res = pd.DataFrame([m_cf, m_cb, m_h], columns=['R-Squared', 'Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)'], index=['Коллаборативная модель (KNN)', 'Контентная модель (Content-Based)', 'Гибридная модель (Hybrid)'])
print(f"Этап 4: Итоговое математическое сравнение. Зафиксирован максимальный оптимум Альфа: {a_v}")
display(res.style.background_gradient(cmap='RdYlGn_r', subset=['Root Mean Squared Error (RMSE)', 'Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)']).background_gradient(cmap='RdYlGn', subset=['R-Squared']))

Используемые файлы данных -> Рейтинги: interactions_small.csv | Метаданные: books_metadata_small.csv
Этап 0: Выполнение базовой предобработки (фильтрация пустот и выбор столбцов)...
Этап 1: Обучение моделей коллаборативной фильтрации (KNN)...
Выполняется обучение алгоритма: KNN Basic (Базовый)
Computing the cosine similarity matrix...
Done computing similarity matrix.
Выполняется обучение алгоритма: KNN With Means (Со средними)
Computing the cosine similarity matrix...
Done computing similarity matrix.
Выполняется обучение алгоритма: KNN With Z-Score (С нормализацией)
Computing the cosine similarity matrix...
Done computing similarity matrix.
Выполняется обучение алгоритма: KNN Baseline (С оптимизацией смещений)
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.


,R-Squared,Root Mean Squared Error (RMSE),Mean Absolute Error (MAE),Mean Squared Error (MSE)
KNN Basic (Базовый),-0.085582,0.977444,0.764470,0.955396
KNN With Means (Со средними),-0.008802,0.942244,0.708035,0.887824
KNN With Z-Score (С нормализацией),-0.016893,0.946015,0.708308,0.894945
KNN Baseline (С оптимизацией смещений),0.067132,0.906088,0.715854,0.820996


Выбранная оптимальная модель коллаборативной фильтрации: KNN Baseline (С оптимизацией смещений)
Этап 2: Извлечение признаков и применение центрирования с Z-нормализацией...
Этап 2.1: Математическое вычисление средних значений и Z-оценок рейтингов...
Этап 3: Выполнение гибридизации и поиск оптимального весового коэффициента Альфа...


,Альфа Коэффициент,R-Squared,Root Mean Squared Error (RMSE),Mean Absolute Error (MAE),Mean Squared Error (MSE)
0,0.00,-0.615692,1.192449,1.004108,1.421934
1,0.05,-0.534799,1.162214,0.981792,1.350742
2,0.10,-0.458828,1.133085,0.959476,1.283882
3,0.15,-0.387777,1.105148,0.937201,1.221352
4,0.20,-0.321648,1.078496,0.914949,1.163153
5,0.25,-0.260440,1.053226,0.892721,1.109286
6,0.30,-0.204154,1.029441,0.870574,1.059749
7,0.35,-0.152788,1.007245,0.848566,1.014543
8,0.40,-0.106344,0.986747,0.827739,0.973669
9,0.45,-0.064821,0.968052,0.808362,0.937125


Этап 4: Итоговое математическое сравнение. Зафиксирован максимальный оптимум Альфа: 0.8500000000000001


,R-Squared,Root Mean Squared Error (RMSE),Mean Absolute Error (MAE),Mean Squared Error (MSE)
Коллаборативная модель (KNN),0.067132,0.906088,0.715854,0.820996
Контентная модель (Content-Based),-0.615692,1.192449,1.004108,1.421934
Гибридная модель (Hybrid),0.090199,0.894816,0.719945,0.800695
